# 00 – Unity Catalog setup
Creates the catalog, layer schemas and volumes for one environment.
Run once per environment with widget `catalog` = `olist_dev` or `olist_prod`.
Safe to re-run (IF NOT EXISTS).

Layering: landing -> bronze -> silver -> gold -> mart_<area>_<report>
(mart_* schemas are created by dbt, one schema per report/consumer)

In [0]:
import re

dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
CATALOG = dbutils.widgets.get("catalog")

# Guard: only our two environments, so a typo can never create a random catalog
if not re.match(r"^olist_(dev|prod)$", CATALOG):
    raise ValueError(f"catalog must be olist_dev or olist_prod, got '{CATALOG}'")

# "olist_prod" -> "prod", used in comments so each catalog describes itself
ENV = CATALOG.split("_")[-1]
print(f"catalog={CATALOG}, env={ENV}")

## Catalog and layer schemas
Catalog = environment, schema = layer.

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
    COMMENT 'Olist lakehouse - {ENV} environment'
""")

# Schema name -> description of the layer
SCHEMAS = {
    "landing": "Raw source files as delivered (volumes)",
    "bronze": "Raw data 1:1 from source + ingestion metadata (Auto Loader, API)",
    "silver": "Cleaned, typed, deduplicated data (dbt staging / intermediate)",
    "gold": "Core star schema: conformed dimensions and facts, single source of truth (dbt)",
}

for schema, comment in SCHEMAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema} COMMENT '{comment}'")
    print(f"OK  schema {CATALOG}.{schema}")

## Volumes
Files live in volumes: landing zone, simulated source system, Auto Loader checkpoints.

In [0]:
# Full volume name -> description
VOLUMES = {
    "landing.raw": "Landing zone for source files (one folder per table)",
    "landing.source": "Simulated source system: full extracts used to replay monthly file drops",
    "bronze.checkpoints": "Auto Loader checkpoints and schema tracking",
}

for volume, comment in VOLUMES.items():
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{volume} COMMENT '{comment}'")
    print(f"OK  volume {CATALOG}.{volume}")